<a href="https://colab.research.google.com/github/Eva360563/MaskArchitectureAnomaly_CourseProject/blob/main/Big_project02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!git clone https://github.com/Eva360563/MaskArchitectureAnomaly_CourseProject.git
%cd MaskArchitectureAnomaly_CourseProject

fatal: destination path 'MaskArchitectureAnomaly_CourseProject' already exists and is not an empty directory.
/content/MaskArchitectureAnomaly_CourseProject


In [11]:
import zipfile, os

zip_path = "/content/drive/MyDrive/Trained_Datasets/Copia di Anomaly_Validation_Datasets.zip"
extract_path = "/content/anomaly_datasets"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Dataset estratti:")
for f in os.listdir(extract_path):
    print(" ", f)

Dataset estratti:
  __MACOSX
  Validation_Dataset


In [12]:
for root, dirs, files in os.walk("/content/anomaly_datasets/Validation_Dataset"):
    level = root.replace("/content/anomaly_datasets/Validation_Dataset", "").count(os.sep)
    if level > 2:
        continue
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level == 2:
        print(f"{indent}  ... ({len(files)} files)")

Validation_Dataset/
  fs_static/
    images/
      ... (30 files)
    labels_masks/
      ... (30 files)
  RoadObsticle21/
    images/
      ... (30 files)
    labels_masks/
      ... (30 files)
  FS_LostFound_full/
    images/
      ... (100 files)
    labels_masks/
      ... (100 files)
  RoadAnomaly/
    images/
      ... (60 files)
    labels_masks/
      ... (60 files)
  RoadAnomaly21/
    images/
      ... (10 files)
    labels_masks/
      ... (10 files)


In [ ]:
!pip install ood-metrics scikit-learn -q
print("Setup ok")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 99.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which 

In [ ]:
#Checking the labels
import glob
import numpy as np
from PIL import Image

DATASETS = {
    "SMIYC_RA21":  "/content/anomaly_datasets/Validation_Dataset/RoadAnomaly21/labels_masks/*.png",
    "SMIYC_RO21":  "/content/anomaly_datasets/Validation_Dataset/RoadObsticle21/labels_masks/*.png",
    "FS_LaF":      "/content/anomaly_datasets/Validation_Dataset/FS_LostFound_full/labels_masks/*.png",
    "FS_Static":   "/content/anomaly_datasets/Validation_Dataset/fs_static/labels_masks/*.png",
    "RoadAnomaly": "/content/anomaly_datasets/Validation_Dataset/RoadAnomaly/labels_masks/*.png",
}

N_SAMPLES = 5

for name, pattern in DATASETS.items():
    print("\n" + "="*60)
    print("DATASET:", name)
    print("="*60)

    paths = glob.glob(pattern)

    if len(paths) == 0:
        print("⚠️ Nessun file trovato!")
        continue

    for i, p in enumerate(paths[:N_SAMPLES]):
        mask = np.array(Image.open(p))
        unique_vals = np.unique(mask)

        print(f"{i+1}. {p.split('/')[-1]}")
        print("   unique values:", unique_vals)

    print("\n")


DATASET: SMIYC_RA21
1. 2.png
   unique values: [  0   1 255]
2. 7.png
   unique values: [  0   1 255]
3. 1.png
   unique values: [  0   1 255]
4. 4.png
   unique values: [  0   1 255]
5. 5.png
   unique values: [  0   1 255]



DATASET: SMIYC_RO21
1. 15.png
   unique values: [  0   1 255]
2. 25.png
   unique values: [  0   1 255]
3. 2.png
   unique values: [  0   1 255]
4. 14.png
   unique values: [  0   1 255]
5. 17.png
   unique values: [  0   1 255]



DATASET: FS_LaF
1. 58.png
   unique values: [  0   1 255]
2. 79.png
   unique values: [  0   1 255]
3. 32.png
   unique values: [  0   1 255]
4. 68.png
   unique values: [  0   1 255]
5. 36.png
   unique values: [  0   1 255]



DATASET: FS_Static
1. 15.png
   unique values: [  0   1 255]
2. 25.png
   unique values: [  0   1 255]
3. 2.png
   unique values: [  0 255]
4. 14.png
   unique values: [  0 255]
5. 17.png
   unique values: [  0 255]



DATASET: RoadAnomaly
1. 58.png
   unique values: [0 2]
2. 32.png
   unique values: [0 2]
3.

In [ ]:
import subprocess

BASE = "/content/anomaly_datasets/Validation_Dataset"
WEIGHTS_DIR = "/content/MaskArchitectureAnomaly_CourseProject/trained_models/"
EVAL_DIR = "/content/MaskArchitectureAnomaly_CourseProject/eval"

DATASETS = {
    "SMIYC_RA21":  f"{BASE}/RoadAnomaly21/images/*.png",
    "SMIYC_RO21":  f"{BASE}/RoadObsticle21/images/*.webp",
    "FS_LaF":      f"{BASE}/FS_LostFound_full/images/*.png",
    "FS_Static":   f"{BASE}/fs_static/images/*.jpg",
    "RoadAnomaly": f"{BASE}/RoadAnomaly/images/*.jpg",
}
METHODS = ["msp", "maxlogit", "maxentropy"]

results = {}
for dataset_name, glob_pattern in DATASETS.items():
    results[dataset_name] = {}
    for method in METHODS:
        print(f"\n▶️ {dataset_name} | {method}")
        out = subprocess.run(
            ["python", "evalAnomaly.py",
             "--input", glob_pattern,
             "--loadDir", WEIGHTS_DIR,
             "--method", method],
            capture_output=True, text=True, cwd=EVAL_DIR
        )
        print(out.stdout[-300:] if out.stdout else "")
        if out.returncode != 0:
            print("ERRORE:", out.stderr[-300:])
        for line in out.stdout.split("\n"):
            if "AUPRC" in line or "FPR" in line:
                print(" →", line)
        results[dataset_name][method] = out.stdout


▶️ SMIYC_RA21 | msp
Modello caricato | Metodo: msp
AuPRC: 29.10%  |  FPR95: 62.55%

 → AuPRC: 29.10%  |  FPR95: 62.55%

▶️ SMIYC_RA21 | maxlogit
Modello caricato | Metodo: maxlogit
AuPRC: 38.32%  |  FPR95: 59.34%

 → AuPRC: 38.32%  |  FPR95: 59.34%

▶️ SMIYC_RA21 | maxentropy
Modello caricato | Metodo: maxentropy
AuPRC: 30.97%  |  FPR95: 62.66%

 → AuPRC: 30.97%  |  FPR95: 62.66%

▶️ SMIYC_RO21 | msp
Modello caricato | Metodo: msp
AuPRC: 2.71%  |  FPR95: 65.22%

 → AuPRC: 2.71%  |  FPR95: 65.22%

▶️ SMIYC_RO21 | maxlogit
Modello caricato | Metodo: maxlogit
AuPRC: 4.63%  |  FPR95: 48.44%

 → AuPRC: 4.63%  |  FPR95: 48.44%

▶️ SMIYC_RO21 | maxentropy
Modello caricato | Metodo: maxentropy
AuPRC: 3.04%  |  FPR95: 65.91%

 → AuPRC: 3.04%  |  FPR95: 65.91%

▶️ FS_LaF | msp
Modello caricato | Metodo: msp
AuPRC: 1.75%  |  FPR95: 50.59%

 → AuPRC: 1.75%  |  FPR95: 50.59%

▶️ FS_LaF | maxlogit
Modello caricato | Metodo: maxlogit
AuPRC: 3.30%  |  FPR95: 45.49%

 → AuPRC: 3.30%  |  FPR95: 45.49%


In [3]:
#CARICAMENTO DEI DATASET PER EOMT
import sys, yaml, importlib
import torch
import numpy as np
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast

# Paths
EOMT_REPO = "/content/MaskArchitectureAnomaly_CourseProject/eomt"
CITYSCAPES_DATA = "/content/drive/MyDrive/Trained_Datasets"
CKPT_CS       = "/content/drive/MyDrive/Trained_Datasets/Copia di eomt_cityscapes.bin"
CKPT_COCO     = "/content/drive/MyDrive/Trained_Datasets/Copia di eomt_coco.bin"
CKPT_FINETUNED = "/content/drive/MyDrive/eomt_finetuned_phase2.bin"

sys.path.insert(0, EOMT_REPO)

import os
os.chdir(EOMT_REPO)

DEVICE = 0
print("✅ Setup ok")

✅ Setup ok


In [4]:
!pip install -q numpy==1.26.4

In [5]:

!pip install lightning -q

In [ ]:
#caricare EoMT che abbiamo GIà USATO PRIMA

def load_eomt(config_path, ckpt_path):
    with open(config_path) as f:
        config = yaml.safe_load(f)

    data_module_name, class_name = config["data"]["class_path"].rsplit(".", 1)
    data_cls = getattr(importlib.import_module(data_module_name), class_name)
    data_kwargs = config["data"].get("init_args", {})
    data = data_cls(path=CITYSCAPES_DATA, batch_size=1, num_workers=0,
                    check_empty_targets=False, **data_kwargs).setup()

    enc_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
    enc_module, enc_classname = enc_cfg["class_path"].rsplit(".", 1)
    enc_cls = getattr(importlib.import_module(enc_module), enc_classname)
    encoder = enc_cls(img_size=data.img_size, **enc_cfg.get("init_args", {}))

    net_cfg = config["model"]["init_args"]["network"]
    net_module, net_classname = net_cfg["class_path"].rsplit(".", 1)
    net_cls = getattr(importlib.import_module(net_module), net_classname)
    net_kwargs = {k: v for k, v in net_cfg["init_args"].items() if k != "encoder"}
    network = net_cls(masked_attn_enabled=False, num_classes=data.num_classes,
                      encoder=encoder, **net_kwargs)

    lit_module, lit_classname = config["model"]["class_path"].rsplit(".", 1)
    lit_cls = getattr(importlib.import_module(lit_module), lit_classname)
    model_kwargs = {k: v for k, v in config["model"]["init_args"].items() if k != "network"}
    if "stuff_classes" in config["data"].get("init_args", {}):
        model_kwargs["stuff_classes"] = config["data"]["init_args"]["stuff_classes"]

    model = lit_cls(img_size=data.img_size, num_classes=data.num_classes,
                    network=network, **model_kwargs).eval().to(DEVICE)

    state_dict = torch.load(ckpt_path, map_location=f"cuda:{DEVICE}", weights_only=True)
    model_state = model.state_dict()
    filtered = {k: v for k, v in state_dict.items()
                if k in model_state and model_state[k].shape == v.shape}
    model.load_state_dict(filtered, strict=False)
    print(f"✅ EoMT caricato da {ckpt_path} | classi: {data.num_classes}")
    return model, data.img_size, data.num_classes

# Carica i 3 modelli
model_cs, img_size_cs, num_classes_cs = load_eomt(
    "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml", CKPT_CS)
model_ft, img_size_ft, _ = load_eomt(
    "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml", CKPT_FINETUNED)

In [7]:
def load_eomt_only(config_path, ckpt_path, num_classes, img_size):
    with open(config_path) as f:
        config = yaml.safe_load(f)
    enc_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
    enc_module, enc_classname = enc_cfg["class_path"].rsplit(".", 1)
    enc_cls = getattr(importlib.import_module(enc_module), enc_classname)
    encoder = enc_cls(img_size=img_size, **enc_cfg.get("init_args", {}))
    net_cfg = config["model"]["init_args"]["network"]
    net_module, net_classname = net_cfg["class_path"].rsplit(".", 1)
    net_cls = getattr(importlib.import_module(net_module), net_classname)
    net_kwargs = {k: v for k, v in net_cfg["init_args"].items() if k != "encoder"}
    network = net_cls(masked_attn_enabled=False, num_classes=num_classes,
                      encoder=encoder, **net_kwargs)
    lit_module, lit_classname = config["model"]["class_path"].rsplit(".", 1)
    lit_cls = getattr(importlib.import_module(lit_module), lit_classname)
    model_kwargs = {k: v for k, v in config["model"]["init_args"].items() if k != "network"}
    if "stuff_classes" in config["data"].get("init_args", {}):
        model_kwargs["stuff_classes"] = config["data"]["init_args"]["stuff_classes"]
    model = lit_cls(img_size=img_size, num_classes=num_classes,
                    network=network, **model_kwargs).eval().to(DEVICE)
    state_dict = torch.load(ckpt_path, map_location=f"cuda:{DEVICE}", weights_only=True)
    model_state = model.state_dict()
    filtered = {k: v for k, v in state_dict.items()
                if k in model_state and model_state[k].shape == v.shape}
    model.load_state_dict(filtered, strict=False)
    print(f"✅ EoMT-COCO caricato | classi: {num_classes}")
    return model

model_coco = load_eomt_only(
    config_path="configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml",
    ckpt_path=CKPT_COCO,
    num_classes=133,
    img_size=(640, 640)
)

✅ EoMT-COCO caricato | classi: 133


In [15]:
#INIZIONE FUNZIONE
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor
from sklearn.metrics import average_precision_score
from ood_metrics import fpr_at_95_tpr

# Transform per le immagini anomaly
anomaly_transform = Compose([
    Resize((1024, 1024), Image.BILINEAR),
    ToTensor(),
])

def load_image_for_eomt(path, size=(1024, 1024)):
    """Carica immagine come uint8 [C, H, W] — formato atteso da EoMT"""
    img = Image.open(path).convert("RGB")
    img = img.resize((size[1], size[0]), Image.BILINEAR)
    img_tensor = torch.from_numpy(np.array(img)).permute(2, 0, 1)  # [3, H, W] uint8
    return img_tensor

def get_eomt_anomaly_score_semantic(model, img_tensor, method, img_size):
    """Per modelli semantic (CS e fine-tuned)"""
    with torch.no_grad(), autocast(device_type="cuda", dtype=torch.float16):
        imgs = [img_tensor.to(DEVICE)]
        img_sizes = [img_tensor.shape[-2:]]
        crops, origins = model.window_imgs_semantic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(mask_logits_per_layer[-1], img_size, mode="bilinear")
        crop_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits_per_layer[-1])
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        per_pixel_logits = logits[0]  # [num_classes, H, W]

    if method == "msp":
        probs = torch.softmax(per_pixel_logits, dim=0)
        score = 1.0 - probs.max(dim=0).values
    elif method == "maxlogit":
        score = 1.0 - per_pixel_logits.max(dim=0).values
    elif method == "maxentropy":
        probs = torch.softmax(per_pixel_logits, dim=0)
        score = -torch.sum(probs * torch.log(probs + 1e-10), dim=0)
    elif method == "rba":
        # RbA: usa mask e class logits separatamente
        mask_probs = torch.sigmoid(mask_logits_per_layer[-1].squeeze(0))  # [Q, H, W]
        mask_probs = F.interpolate(mask_probs.unsqueeze(0), img_size, mode="bilinear").squeeze(0)
        class_probs = torch.softmax(class_logits_per_layer[-1].squeeze(0), dim=-1)[:, :-1]  # [Q, C]
        max_class_prob = class_probs.max(dim=-1).values  # [Q]
        query_scores = max_class_prob[:, None, None] * mask_probs  # [Q, H, W]
        score = 1.0 - query_scores.max(dim=0).values  # [H, W]

    return score.cpu().float().numpy()


def get_eomt_anomaly_score_panoptic(model, img_tensor, method, img_size):
    """Per modello COCO (panoptic)"""
    with torch.no_grad(), autocast(device_type="cuda", dtype=torch.float16):
        imgs = [img_tensor.to(DEVICE)]
        img_sizes = [img_tensor.shape[-2:]]
        transformed = model.resize_and_pad_imgs_instance_panoptic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(transformed)
        mask_logits = F.interpolate(mask_logits_per_layer[-1], model.img_size, mode="bilinear")
        mask_logits = model.revert_resize_and_pad_logits_instance_panoptic(mask_logits, img_sizes)
        # Converti in per-pixel logits semantici
        per_pixel_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        per_pixel_logits = model.revert_window_logits_semantic(
            per_pixel_logits, [[0,0]], img_sizes
        )
        logits = per_pixel_logits[0]  # [num_classes, H, W]

    if method == "msp":
        probs = torch.softmax(logits, dim=0)
        score = 1.0 - probs.max(dim=0).values
    elif method == "maxlogit":
        score = 1.0 - logits.max(dim=0).values
    elif method == "maxentropy":
        probs = torch.softmax(logits, dim=0)
        score = -torch.sum(probs * torch.log(probs + 1e-10), dim=0)
    elif method == "rba":
        mask_probs = torch.sigmoid(mask_logits.squeeze(0))  # [Q, H, W]
        class_probs = torch.softmax(class_logits_per_layer[-1].squeeze(0), dim=-1)[:, :-1]
        max_class_prob = class_probs.max(dim=-1).values
        query_scores = max_class_prob[:, None, None] * mask_probs
        score = 1.0 - query_scores.max(dim=0).values

    return score.cpu().float().numpy()

In [17]:
import glob

def evaluate_eomt_on_dataset(model, glob_pattern, method, is_panoptic=False, img_size=(1024,1024)):
    anomaly_score_list = []
    ood_gts_list = []
    target_transform = Compose([Resize((1024, 1024), Image.NEAREST)])

    for path in glob.glob(glob_pattern):
        # ✅ carica come uint8
        img = load_image_for_eomt(path, size=img_size)

        if is_panoptic:
            score = get_eomt_anomaly_score_panoptic(model, img, method, img_size)
        else:
            score = get_eomt_anomaly_score_semantic(model, img, method, img_size)

        pathGT = path.replace("images", "labels_masks")
        if "RoadObsticle21" in pathGT: pathGT = pathGT.replace("webp", "png")
        if "fs_static" in pathGT: pathGT = pathGT.replace("jpg", "png")
        if "RoadAnomaly" in pathGT: pathGT = pathGT.replace("jpg", "png")

        ood_gts = np.array(target_transform(Image.open(pathGT)))

        if "RoadAnomaly" in pathGT and "RoadAnomaly21" not in pathGT:
            ood_gts = np.where((ood_gts == 2), 1, ood_gts)

        if 1 not in np.unique(ood_gts):
            continue

        anomaly_score_list.append(score)
        ood_gts_list.append(ood_gts)
        torch.cuda.empty_cache()

    if len(ood_gts_list) == 0:
        print("  Nessuna immagine valida!")
        return 0.0, 100.0

    ood_gts = np.array(ood_gts_list)
    anomaly_scores = np.array(anomaly_score_list)

    ood_out = anomaly_scores[ood_gts == 1]
    ind_out = anomaly_scores[ood_gts == 0]
    val_out = np.concatenate((ind_out, ood_out))
    val_label = np.concatenate((np.zeros(len(ind_out)), np.ones(len(ood_out))))

    prc_auc = average_precision_score(val_label, val_out)
    fpr = fpr_at_95_tpr(val_out, val_label)
    return prc_auc*100, fpr*100

In [18]:
BASE = "/content/anomaly_datasets/Validation_Dataset"
DATASETS = {

    "SMIYC_RA21":  f"{BASE}/RoadAnomaly21/images/*.png",
    "SMIYC_RO21":  f"{BASE}/RoadObsticle21/images/*.webp",
    "FS_LaF":      f"{BASE}/FS_LostFound_full/images/*.png",
    "FS_Static":   f"{BASE}/fs_static/images/*.jpg",
    "RoadAnomaly": f"{BASE}/RoadAnomaly/images/*.jpg",
}

MODELS = {
    "EoMT-CS":        (model_cs,   False, (1024,1024)),
    "EoMT-COCO":      (model_coco, True,  (640,640)),
    "EoMT-Finetuned": (model_ft,   False, (1024,1024)),
}
METHODS_SEMANTIC  = ["msp", "maxlogit", "maxentropy", "rba"]
METHODS_PANOPTIC  = ["msp", "maxlogit", "maxentropy", "rba"]

results = {}
for model_name, (model, is_panoptic, img_size) in MODELS.items():
    results[model_name] = {}
    for method in METHODS_SEMANTIC:
        results[model_name][method] = {}
        print(f"\n{'='*50}")
        print(f"▶ {model_name} | {method}")
        for dataset_name, glob_pattern in DATASETS.items():
            auprc, fpr = evaluate_eomt_on_dataset(
                model, glob_pattern, method,
                is_panoptic=is_panoptic, img_size=img_size
            )
            results[model_name][method][dataset_name] = (auprc, fpr)
            print(f"  {dataset_name}: AuPRC={auprc:.2f}% FPR95={fpr:.2f}%")

print("\n✅ Evaluation completata!")


▶ EoMT-CS | msp
  SMIYC_RA21: AuPRC=74.37% FPR95=33.73%
  SMIYC_RO21: AuPRC=90.11% FPR95=0.55%
  FS_LaF: AuPRC=25.42% FPR95=14.90%
  FS_Static: AuPRC=54.32% FPR95=34.91%
  RoadAnomaly: AuPRC=75.67% FPR95=19.34%

▶ EoMT-CS | maxlogit
  SMIYC_RA21: AuPRC=73.99% FPR95=33.90%
  SMIYC_RO21: AuPRC=90.05% FPR95=0.58%
  FS_LaF: AuPRC=25.44% FPR95=14.65%
  FS_Static: AuPRC=54.41% FPR95=34.57%
  RoadAnomaly: AuPRC=75.15% FPR95=19.09%

▶ EoMT-CS | maxentropy
  SMIYC_RA21: AuPRC=76.86% FPR95=33.60%
  SMIYC_RO21: AuPRC=90.00% FPR95=0.68%
  FS_LaF: AuPRC=28.02% FPR95=14.72%
  FS_Static: AuPRC=52.72% FPR95=34.82%
  RoadAnomaly: AuPRC=77.62% FPR95=18.83%

▶ EoMT-CS | rba
  SMIYC_RA21: AuPRC=77.97% FPR95=34.94%
  SMIYC_RO21: AuPRC=90.23% FPR95=0.36%


KeyboardInterrupt: 